In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from uncertainties import ufloat
import uncertainties.umath as umath
from uncertainties import correlated_values

In [11]:
plt.rcParams["text.usetex"] = True
plt.rcParams['font.size'] = 16
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['axes.titlesize'] = 17
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 14

# Linien und Marker
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['lines.markersize'] = 8
plt.rcParams['errorbar.capsize'] = 5

# DPI und Qualität
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['figure.constrained_layout.use'] = True

# Achsen-Styling
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['xtick.major.width'] = 1.2
plt.rcParams['ytick.major.width'] = 1.2
plt.rcParams['xtick.major.size'] = 6
plt.rcParams['ytick.major.size'] = 6

In [12]:
# Konfiguration
DATA_FILE = Path("data/xst.xlsx")

# Schwellenwert für die Intensität (R-Wert)
R_THRESHOLD = 80

SHEET_TO_U_KV = {
    "A6.1": 35.0,
    "A6.2": 32.5,
    "A6.3": 30.0,
    "A6.4": 28.0,
    "A6.5": 26.0,
    "A6.6": 24.0,
}

FIT_WINDOWS_DEG = {
    35.0: (5, 6.5),
    32.5: (5.5, 7.0),
    30.0: (6, 7.5),
    28.0: (6.5, 8.0),
    26.0: (7, 8.5),
    24.0: (7.5, 8.5),
}


In [13]:
def add_linear_fit_to_ax(ax, x, y, u_kv, fit_window=None):
    """
    Fügt einen linearen Fit (ax+b) zu einem Plot hinzu und zeigt die Gleichung an.

    Parameters:
    -----------
    ax : matplotlib axis
        Die Achse, zu der der Fit hinzugefügt werden soll
    x : array-like
        X-Daten (theta)
    y : array-like
        Y-Daten (Intensität)
    u_kv : float
        Spannungswert für die Legende
    fit_window : tuple, optional
        (x_min, x_max) für den Fit-Bereich. Wenn None, wird der gesamte Bereich verwendet.
    """
    # Daten in den Fit-Bereich filtern
    if fit_window is not None:
        x_min, x_max = fit_window
        # Finde den Index, wo R > R_THRESHOLD ist
        idx_r_gt_threshold = np.where(y > R_THRESHOLD)[0]
        if len(idx_r_gt_threshold) > 0:
            # Nutze den ersten Index wo R > R_THRESHOLD als obere Grenze
            x_max = min(x_max, x[idx_r_gt_threshold[0]])

        mask = (x >= x_min) & (x <= x_max)
        x_fit = x[mask]
        y_fit = y[mask]
    else:
        x_fit = x
        y_fit = y

    # Linearen Fit durchführen: y = a*x + b
    coefficients = np.polyfit(x_fit, y_fit, 1)
    a = coefficients[0]
    b = coefficients[1]

    # Fit-Kurve berechnen nur im Fit-Fenster
    x_line = np.linspace(x_fit.min(), x_fit.max(), 100)
    y_line = a * x_line + b

    # Fit-Kurve plotten
    ax.plot(x_line, y_line, 'r--', linewidth=2, label=f'Fit: y = {a:.2f}x + {b:.2f}')

    # Gleichung in den Plot schreiben
    equation_text = f'y = {a:.2f}x + {b:.2f}'
    ax.text(0.05, 0.95, equation_text, transform=ax.transAxes,
             verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.legend()


In [14]:
# please plot the data and each sheet in a separate subplot, with the x-axis being the angle (theta) and the y-axis being the intensity (I).
# Make sure to label each subplot with the corresponding U_KV value from the SHEET_TO_U_KV dictionary, and include appropriate axis labels and titles.
# increase legend font size

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(16, 20))

# Gruppe 1: U = 35, 32.5, 30 kV (erste 3)
# Gruppe 2: U = 28, 26, 24 kV (letzte 3)
groups = [
    [("A6.1", 35.0), ("A6.2", 32.5), ("A6.3", 30.0)],
    [("A6.4", 28.0), ("A6.5", 26.0), ("A6.6", 24.0)]
]

colors = ['green', 'orange', 'red', 'purple', 'blue', 'brown']
color_idx = 0

# Sammle Fit-Parameter für DataFrame
fit_params_data = []

for group_idx, group in enumerate(groups):
    ax = axes[group_idx]

    for sheet_name, u_kv in group:
        df = pd.read_excel(DATA_FILE, sheet_name=sheet_name)

        # Linearen Fit hinzufügen mit FIT_WINDOWS
        fit_window = FIT_WINDOWS_DEG.get(u_kv)

        # Filter die Daten basierend auf dem Fit-Fenster und R > R_THRESHOLD
        x_data = df["beta"].values
        y_data = df["R"].values

        if fit_window is not None:
            x_min, x_max = fit_window
            # Finde den Index, wo R > R_THRESHOLD ist
            idx_r_gt_threshold = np.where(y_data > R_THRESHOLD)[0]
            if len(idx_r_gt_threshold) > 0:
                x_max = min(x_max, x_data[idx_r_gt_threshold[0]])

            mask = (x_data >= x_min) & (x_data <= x_max)
            x_plot = x_data[mask]
            y_plot = y_data[mask]
        else:
            x_plot = x_data
            y_plot = y_data

        # Plotte nur die gefilterten Daten
        current_color = colors[color_idx]
        ax.plot(x_plot, y_plot, marker='o', linestyle='-', label=f'U={u_kv} kV',
                color=current_color, linewidth=2, markersize=5)

        # Fit-Linie berechnen und plotten
        if fit_window is not None:
            x_min, x_max = fit_window
            idx_r_gt_threshold = np.where(y_data > R_THRESHOLD)[0]
            if len(idx_r_gt_threshold) > 0:
                x_max = min(x_max, x_data[idx_r_gt_threshold[0]])

            mask = (x_data >= x_min) & (x_data <= x_max)
            x_fit = x_data[mask]
            y_fit = y_data[mask]
        else:
            x_fit = x_data
            y_fit = y_data

        # Polyfit
        coefficients = np.polyfit(x_fit, y_fit, 1)
        a = coefficients[0]
        b = coefficients[1]

        # Berechne Fit-Unsicherheiten
        z = np.polyfit(x_fit, y_fit, 1, cov=True)
        cov_matrix = z[1]
        da = np.sqrt(cov_matrix[0, 0])  # Unsicherheit in Steigung
        db = np.sqrt(cov_matrix[1, 1])  # Unsicherheit in Intercept

        # Berechne θ₀ (Nullstelle der Fitlinie: y = 0 => x = -b/a)
        theta0 = -b / a

        # Speichere Parameter für DataFrame
        param_idx = color_idx + 1  # a1-a6, b1-b6
        fit_params_data.append({
            'Parameter': f'$a_{{{param_idx}}}$',
            'U_kV': u_kv,
            'Value': a,
            'Uncertainty': da,
            'Value_with_unc': ufloat(a, da)
        })
        fit_params_data.append({
            'Parameter': f'$b_{{{param_idx}}}$',
            'U_kV': u_kv,
            'Value': b,
            'Uncertainty': db,
            'Value_with_unc': ufloat(b, db)
        })

        # Fit-Kurve berechnen von θ₀ bis zum Ende des Fit-Fensters
        x_line = np.linspace(theta0, x_fit.max(), 100)
        y_line = a * x_line + b

        # Fit-Kurve plotten mit den tatsächlichen Werten und Unsicherheiten
        fit_label = f'Fit U={u_kv} kV: $y = ({a:.1f} \\pm {da:.1f})x + ({b:.0f} \\pm {db:.0f})$'
        ax.plot(x_line, y_line, '--', linewidth=2, color=current_color, alpha=0.7, label=fit_label)

        color_idx += 1

    ax.set_xlabel(r'Winkel $\theta\ /\ ^\circ$')
    ax.set_ylabel(r'Zählrate R_z / 1/s')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()


# save with proper name as svg
plt.savefig("xst_intensity_vs_theta_with_fits.svg", format='svg', bbox_inches='tight', dpi=150)
plt.show()


RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpwpgigajs 9f558406edbc51514263636e58d51433.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./9f558406edbc51514263636e58d51433.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025basic/texmf-dist/tex/latex/base/article.cls
Document Class: article 2024/06/29 v1.4n Standard LaTeX document class
(/usr/local/texlive/2025basic/texmf-dist/tex/latex/base/size10.clo))

! LaTeX Error: File `type1cm.sty' not found.

Type X to quit or <RETURN> to proceed,
or enter new name. (Default extension: sty)

Enter file name: 
! Emergency stop.
<read *> 
         
l.7 \usepackage
               {type1ec}^^M
No pages of output.
Transcript written on tmpwpgigajs/9f558406edbc51

Error in callback <function _draw_all_if_interactive at 0x1111c7310> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpd3zv11_d 9f558406edbc51514263636e58d51433.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./9f558406edbc51514263636e58d51433.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025basic/texmf-dist/tex/latex/base/article.cls
Document Class: article 2024/06/29 v1.4n Standard LaTeX document class
(/usr/local/texlive/2025basic/texmf-dist/tex/latex/base/size10.clo))

! LaTeX Error: File `type1cm.sty' not found.

Type X to quit or <RETURN> to proceed,
or enter new name. (Default extension: sty)

Enter file name: 
! Emergency stop.
<read *> 
         
l.7 \usepackage
               {type1ec}^^M
No pages of output.
Transcript written on tmpd3zv11_d/9f558406edbc51

RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpwvay5iv1 9f558406edbc51514263636e58d51433.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./9f558406edbc51514263636e58d51433.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025basic/texmf-dist/tex/latex/base/article.cls
Document Class: article 2024/06/29 v1.4n Standard LaTeX document class
(/usr/local/texlive/2025basic/texmf-dist/tex/latex/base/size10.clo))

! LaTeX Error: File `type1cm.sty' not found.

Type X to quit or <RETURN> to proceed,
or enter new name. (Default extension: sty)

Enter file name: 
! Emergency stop.
<read *> 
         
l.7 \usepackage
               {type1ec}^^M
No pages of output.
Transcript written on tmpwvay5iv1/9f558406edbc51

<Figure size 2400x3000 with 2 Axes>

In [15]:
# Erstelle DataFrame mit Fit-Parametern
fit_params_df = pd.DataFrame(fit_params_data)

# Formatiere die Ausgabe
print("=" * 100)
print("FIT-PARAMETER a1-a6 UND b1-b6 MIT UNSICHERHEITEN")
print("=" * 100)
print("\nDetaillierte Tabelle:")
print(fit_params_df[['Parameter', 'U_kV', 'Value_with_unc']])

# Erstelle eine kompaktere Ansicht
print("\n" + "=" * 100)
print("KOMPAKTE ÜBERSICHT:")
print("=" * 100)
for idx, row in fit_params_df.iterrows():
    print(f"{row['Parameter']:4s} (U={row['U_kV']:5.1f} kV): {row['Value_with_unc']}")

# Exportiere als separate Tabelle für a und b Parameter
print("\n" + "=" * 100)
print("GETRENNTE TABELLEN:")
print("=" * 100)
a_params = fit_params_df[fit_params_df['Parameter'].str.startswith('a')].copy()
b_params = fit_params_df[fit_params_df['Parameter'].str.startswith('b')].copy()

print("\nSteigungen (a1-a6):")
print(a_params[['Parameter', 'U_kV', 'Value', 'Uncertainty']])

print("\nAchsenabschnitte (b1-b6):")
print(b_params[['Parameter', 'U_kV', 'Value', 'Uncertainty']])

fit_params_df

FIT-PARAMETER a1-a6 UND b1-b6 MIT UNSICHERHEITEN

Detaillierte Tabelle:
   Parameter  U_kV      Value_with_unc
0    $a_{1}$  35.0     (2.1+/-0.5)e+02
1    $b_{1}$  35.0  (-1.02+/-0.28)e+03
2    $a_{2}$  32.5     (3.3+/-0.6)e+02
3    $b_{2}$  32.5  (-1.80+/-0.34)e+03
4    $a_{3}$  30.0            250+/-28
5    $b_{3}$  30.0  (-1.49+/-0.17)e+03
6    $a_{4}$  28.0     (3.2+/-0.4)e+02
7    $b_{4}$  28.0  (-2.09+/-0.28)e+03
8    $a_{5}$  26.0            281+/-14
9    $b_{5}$  26.0  (-1.95+/-0.10)e+03
10   $a_{6}$  24.0            179+/-14
11   $b_{6}$  24.0  (-1.34+/-0.11)e+03

KOMPAKTE ÜBERSICHT:
$a_{1}$ (U= 35.0 kV): (2.1+/-0.5)e+02
$b_{1}$ (U= 35.0 kV): (-1.02+/-0.28)e+03
$a_{2}$ (U= 32.5 kV): (3.3+/-0.6)e+02
$b_{2}$ (U= 32.5 kV): (-1.80+/-0.34)e+03
$a_{3}$ (U= 30.0 kV): 250+/-28
$b_{3}$ (U= 30.0 kV): (-1.49+/-0.17)e+03
$a_{4}$ (U= 28.0 kV): (3.2+/-0.4)e+02
$b_{4}$ (U= 28.0 kV): (-2.09+/-0.28)e+03
$a_{5}$ (U= 26.0 kV): 281+/-14
$b_{5}$ (U= 26.0 kV): (-1.95+/-0.10)e+03
$a_{6}$ (U= 24.0 kV

,Parameter,U_kV,Value,Uncertainty,Value_with_unc
0,$a_{1}$,35.0,205.300000,53.458676,(2.1+/-0.5)e+02
1,$b_{1}$,35.0,-1016.720000,275.377049,(-1.02+/-0.28)e+03
2,$a_{2}$,32.5,328.900000,60.493553,(3.3+/-0.6)e+02
3,$b_{2}$,32.5,-1800.010000,341.855488,(-1.80+/-0.34)e+03
4,$a_{3}$,30.0,249.900000,27.554854,250+/-28
5,$b_{3}$,30.0,-1490.860000,169.490355,(-1.49+/-0.17)e+03
6,$a_{4}$,28.0,324.700000,43.128065,(3.2+/-0.4)e+02
7,$b_{4}$,28.0,-2092.290000,284.667011,(-2.09+/-0.28)e+03
8,$a_{5}$,26.0,280.810000,14.000525,281+/-14
9,$b_{5}$,26.0,-1947.589000,100.115991,(-1.95+/-0.10)e+03


In [33]:

# Unsicherheiten
THETA_UNCERTAINTY = 0.28  # Grad
U_UNCERTAINTY = 0.1  # kV
I_UNCERTAINTY = 20e-6  # A (20 µA)

# Physikalische Konstanten
h = 6.62607015e-34  # J·s (Planck-Konstante)
c = 299792458  # m/s (Lichtgeschwindigkeit)
e = 1.602176634e-19  # C (Elementarladung)

# Gemittelte Gitterkonstante a (401,97 ± 1,99) pm

d_bragg = 402.0e-12 / 2  # m
d_bragg_uncertainty = 2.0e-12 / 2  # m


In [34]:
def calculate_theta0_with_uncertainty(sheet_name, u_kv, fit_window):
    """
    Berechnet θ₀ durch Extrapolation des linearen Fits mit Unsicherheiten.

    Parameters:
    -----------
    sheet_name : str
        Name des Excel-Sheets
    u_kv : float
        Hochspannung in kV
    fit_window : tuple
        (x_min, x_max) für den Fit-Bereich

    Returns:
    --------
    dict mit θ₀, λ₀, h und allen Unsicherheiten
    """
    df = pd.read_excel(DATA_FILE, sheet_name=sheet_name)
    x_data = df["beta"].values
    y_data = df["R"].values

    # Filtere Daten für Fit
    if fit_window is not None:
        x_min, x_max = fit_window
        idx_r_gt_threshold = np.where(y_data > R_THRESHOLD)[0]
        if len(idx_r_gt_threshold) > 0:
            x_max = min(x_max, x_data[idx_r_gt_threshold[0]])

        mask = (x_data >= x_min) & (x_data <= x_max)
        x_fit = x_data[mask]
        y_fit = y_data[mask]
    else:
        x_fit = x_data
        y_fit = y_data

    # Polyfit mit Unsicherheitsberechnung
    coefficients = np.polyfit(x_fit, y_fit, 1)
    a = coefficients[0]
    b = coefficients[1]

    # # Berechne Fit-Unsicherheiten mittels polyfit_cov
    # z = np.polyfit(x_fit, y_fit, 1, cov=True)
    # cov_matrix = z[1]
    # da = np.sqrt(cov_matrix[0, 0])  # Unsicherheit in Steigung
    # db = np.sqrt(cov_matrix[1, 1])  # Unsicherheit in Intercept

    # # θ₀ bestimmen durch Extrapolation: 0 = a*θ₀ + b => θ₀ = -b/a
    # theta0_nominal = -b / a

    # # Unsicherheit in θ₀ durch Fehlerfortpflanzung
    # # dθ₀/da = b/a², dθ₀/db = -1/a
    # d_theta0_da = b / (a**2)
    # d_theta0_db = -1 / a
    # dtheta0_fit = np.sqrt((d_theta0_da * da)**2 + (d_theta0_db * db)**2)

    # # Winkel-Unsicherheit
    # dtheta0_angle = THETA_UNCERTAINTY

    # # Gesamtunsicherheit in θ₀ (kombiniert Fit und Winkel)
    # dtheta0_total = np.sqrt(dtheta0_fit**2 + dtheta0_angle**2)

    # # θ₀ mit Unsicherheit als ufloat
    # theta0 = ufloat(theta0_nominal, dtheta0_total)
    
    coefficients, cov_matrix = np.polyfit(x_fit, y_fit, 1, cov=True)
    
    params = correlated_values(coefficients, cov_matrix)
    a_u, b_u = params  
    
    # theta0_fit als ufloat (enthält jetzt die statistische Unsicherheit des Fits)
    theta0_fit_u = -b_u / a_u

    dtheta0_angle = THETA_UNCERTAINTY
    
    theta0_nominal = theta0_fit_u.nominal_value
    dtheta0_fit = theta0_fit_u.std_dev

    dtheta0_total = np.sqrt(dtheta0_fit**2 + dtheta0_angle**2)

    theta0 = ufloat(theta0_nominal, dtheta0_total)

    # d_bragg mit Unsicherheit als ufloat
    d_bragg_ufloat = ufloat(d_bragg, d_bragg_uncertainty)

    # Bragg-Bedingung: λ₀ = 2*d*sin(θ₀) (für 1. Ordnung n=1)
    # Nutze ufloat für automatische Fehlerfortpflanzung
    theta0_rad_ufloat = theta0 * np.pi / 180  # Konvertiere zu Radiant mit Unsicherheit

    # Berechne λ₀ mit automatischer Fehlerfortpflanzung
    lambda0 = 2 * d_bragg_ufloat * umath.sin(theta0_rad_ufloat)


    # Planck-Konstante aus λ₀ = h*c/(e*U)
    # h = λ₀*e*U/c
    U_joule = ufloat(u_kv * 1000, U_UNCERTAINTY * 1000)  # Konvertiere zu V mit Unsicherheit

    h_calculated = (lambda0 * e * U_joule) / c

    # Konvertiere h in eV·s (durch Division durch e)
    h_calculated_eV = h_calculated / e

    return {
        'U_kV': u_kv,
        'theta0_deg': theta0,
        'theta0_nominal': theta0_nominal,
        'theta0_uncertainty': dtheta0_total,
        'lambda0_m': lambda0,
        'lambda0_nominal': float(lambda0.nominal_value),
        'lambda0_uncertainty': float(lambda0.std_dev),
        'h_calculated': h_calculated,
        'h_nominal': float(h_calculated.nominal_value),
        'h_uncertainty': float(h_calculated.std_dev),
        'h_calculated_eV': h_calculated_eV,
        'h_nominal_eV': float(h_calculated_eV.nominal_value),
        'h_uncertainty_eV': float(h_calculated_eV.std_dev),
        'a': a,
        'b': b,
        'da': da,
        'db': db,
    }


In [35]:
# Berechne θ₀ für alle Messungen
results = []
for sheet_name, u_kv in SHEET_TO_U_KV.items():
    fit_window = FIT_WINDOWS_DEG.get(u_kv)
    result = calculate_theta0_with_uncertainty(sheet_name, u_kv, fit_window)
    results.append(result)

# Erstelle DataFrame
results_df = pd.DataFrame(results)

# Formatiere Ausgabe
print("=" * 100)
print("THETA_0 BESTIMMUNG MIT UNSICHERHEITEN")
print("=" * 100)
print("\nDetaillierte Ergebnisse (J·s):")
print(results_df[['U_kV', 'theta0_deg', 'theta0_uncertainty', 'lambda0_m', 'h_calculated']])

print("\nDetaillierte Ergebnisse (eV·s):")
print(results_df[['U_kV', 'h_calculated_eV']])

print("\n" + "=" * 100)
print("PLANCK-KONSTANTE h (J·s und eV·s)")
print("=" * 100)
for idx, row in results_df.iterrows():
    h_str = str(row['h_calculated'])
    h_eV_str = str(row['h_calculated_eV'])
    print(f"U = {row['U_kV']:5.1f} kV: h = {h_str:40s} J·s")
    print(f"              h = {h_eV_str:40s} eV·s")
    print()

# Gewichteter Mittelwert
h_values = results_df['h_calculated'].values
h_mean = np.mean([float(h.nominal_value) for h in h_values])
h_weights = 1 / np.array([float(h.std_dev)**2 for h in h_values])
h_weighted_mean = np.sum(h_weights * np.array([float(h.nominal_value) for h in h_values])) / np.sum(h_weights)
h_weighted_uncertainty = np.sqrt(1 / np.sum(h_weights))

print(f"\nGewichteter Mittelwert: h = ({h_weighted_mean:.4e} ± {h_weighted_uncertainty:.4e}) J·s")
print(f"Theoretischer Wert:    h = {6.62607015e-34:.4e} J·s")

# DataFrame exportieren
print("\n" + "=" * 100)
print("PANDAS DATAFRAME:")
print("=" * 100)
print(results_df)

results_df


THETA_0 BESTIMMUNG MIT UNSICHERHEITEN

Detaillierte Ergebnisse (J·s):
   U_kV   theta0_deg  theta0_uncertainty          lambda0_m       h_calculated
0  35.0  4.95+/-0.29            0.286175  (3.47+/-0.20)e-11    (6.5+/-0.4)e-34
1  32.5  5.47+/-0.28            0.282639  (3.83+/-0.20)e-11  (6.66+/-0.35)e-34
2  30.0  5.97+/-0.28            0.281006  (4.18+/-0.20)e-11  (6.70+/-0.32)e-34
3  28.0  6.44+/-0.28            0.280977  (4.51+/-0.20)e-11  (6.75+/-0.30)e-34
4  26.0  6.94+/-0.28            0.280259  (4.85+/-0.20)e-11  (6.75+/-0.27)e-34
5  24.0  7.50+/-0.28            0.280945  (5.25+/-0.20)e-11  (6.73+/-0.25)e-34

Detaillierte Ergebnisse (eV·s):
   U_kV    h_calculated_eV
0  35.0  (4.05+/-0.23)e-15
1  32.5  (4.16+/-0.22)e-15
2  30.0  (4.18+/-0.20)e-15
3  28.0  (4.21+/-0.18)e-15
4  26.0  (4.21+/-0.17)e-15
5  24.0  (4.20+/-0.16)e-15

PLANCK-KONSTANTE h (J·s und eV·s)
U =  35.0 kV: h = (6.5+/-0.4)e-34                          J·s
              h = (4.05+/-0.23)e-15                      

,U_kV,theta0_deg,theta0_nominal,theta0_uncertainty,lambda0_m,lambda0_nominal,lambda0_uncertainty,h_calculated,h_nominal,h_uncertainty,h_calculated_eV,h_nominal_eV,h_uncertainty_eV,a,b,da,db
0,35.0,4.95+/-0.29,4.952362,0.286175,(3.47+/-0.20)e-11,3.470363e-11,2.007808e-12,(6.5+/-0.4)e-34,6.491315e-34,3.760182e-35,(4.05+/-0.23)e-15,4.051560e-15,2.346921e-16,205.30,-1016.720000,13.746051,106.557755
1,32.5,5.47+/-0.28,5.472818,0.282639,(3.83+/-0.20)e-11,3.834015e-11,1.983214e-12,(6.66+/-0.35)e-34,6.659274e-34,3.450719e-35,(4.16+/-0.22)e-15,4.156392e-15,2.153770e-16,328.90,-1800.010000,13.746051,106.557755
2,30.0,5.97+/-0.28,5.965826,0.281006,(4.18+/-0.20)e-11,4.178198e-11,1.971910e-12,(6.70+/-0.32)e-34,6.698845e-34,3.169411e-35,(4.18+/-0.20)e-15,4.181090e-15,1.978191e-16,249.90,-1490.860000,13.746051,106.557755
3,28.0,6.44+/-0.28,6.443763,0.280977,(4.51+/-0.20)e-11,4.511564e-11,1.971763e-12,(6.75+/-0.30)e-34,6.751104e-34,2.960381e-35,(4.21+/-0.18)e-15,4.213708e-15,1.847724e-16,324.70,-2092.290000,13.746051,106.557755
4,26.0,6.94+/-0.28,6.935611,0.280259,(4.85+/-0.20)e-11,4.854305e-11,1.966857e-12,(6.75+/-0.27)e-34,6.745126e-34,2.745262e-35,(4.21+/-0.17)e-15,4.209977e-15,1.713458e-16,280.81,-1947.589000,13.746051,106.557755
5,24.0,7.50+/-0.28,7.504234,0.280945,(5.25+/-0.20)e-11,5.250098e-11,1.971667e-12,(6.73+/-0.25)e-34,6.733927e-34,2.544434e-35,(4.20+/-0.16)e-15,4.202986e-15,1.588111e-16,178.70,-1341.006667,13.746051,106.557755


In [23]:
# Bitte berechne den mittelwert von h für eV mit unsicherheiten
h_eV_values = results_df['h_calculated_eV'].values
h_eV_mean = np.mean([float(h.nominal_value) for h in h_eV_values])
h_eV_weights = 1 / np.array([float(h.std_dev)**2 for h in h_eV_values])
h_eV_weighted_mean = np.sum(h_eV_weights * np.array([float(h.nominal_value) for h in h_eV_values])) / np.sum(h_eV_weights)
h_eV_weighted_uncertainty = np.sqrt(1 / np.sum(h_eV_weights))
# display data properly
print(f"\nGewichteter Mittelwert: h = ({h_eV_weighted_mean:.4e} ± {h_eV_weighted_uncertainty:.4e}) eV·s")
print(f"Theoretischer Wert:    h = {6.62607015e-34 / e:.4e} eV·s")



Gewichteter Mittelwert: h = (4.1988e-15 ± 2.3957e-16) eV·s
Theoretischer Wert:    h = 4.1357e-15 eV·s
